In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [4]:
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_3.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_3.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_3.csv')

In [5]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [6]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

30650
30650
30650


In [7]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.4882], device='cuda:0')
torch.Size([12289])


In [8]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([-0.0044, -0.0028], device='cuda:0')
torch.Size([2])


In [9]:
dataset = TensorDataset(X_tensor, y_tensor)
data_loader = DataLoader(dataset, batch_size=512, shuffle=False)

In [10]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_size=12289, hidden_size_1=1024, hidden_size_2=512, hidden_size_3=512,
                 output_size=2):
        super(FeedforwardNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.fc3 = nn.Linear(hidden_size_2, hidden_size_3)
        self.fc4 = nn.Linear(hidden_size_3, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, input_data):
        out = self.fc1(input_data)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        out = self.tanh(out)
        return out

In [11]:
model = FeedforwardNet().cuda()

In [12]:
# Определение функции потерь и оптимизатора
criterion = nn.MSELoss()  # Функция потерь для регрессионной задачи
optimizer = optim.Adam(model.parameters(), lr=0.000001)  # Оптимизатор Adam

# Обучение модели
num_epochs = 100
for epoch in range(num_epochs):
    for batch_x, batch_y in data_loader:  # Итерация по батчам данных
        optimizer.zero_grad()  # Обнуление градиентов

        outputs = model(batch_x.unsqueeze(0))  # Передача входных данных через модель
        loss = criterion(outputs, batch_y)  # Вычисление потерь

        loss.backward()  # Обратное распространение ошибки
        optimizer.step()  # Обновление весов модели

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([512, 2])) that is different to the input size (torch.Size([1, 512, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([442, 2])) that is different to the input size (torch.Size([1, 442, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/100], Loss: 0.000130593
Epoch [2/100], Loss: 0.000104218
Epoch [3/100], Loss: 0.000084269
Epoch [4/100], Loss: 0.000069975
Epoch [5/100], Loss: 0.000059651
Epoch [6/100], Loss: 0.000052011
Epoch [7/100], Loss: 0.000046257
Epoch [8/100], Loss: 0.000041790
Epoch [9/100], Loss: 0.000038281
Epoch [10/100], Loss: 0.000035455
Epoch [11/100], Loss: 0.000033109
Epoch [12/100], Loss: 0.000031093
Epoch [13/100], Loss: 0.000029330
Epoch [14/100], Loss: 0.000027769
Epoch [15/100], Loss: 0.000026371
Epoch [16/100], Loss: 0.000025110
Epoch [17/100], Loss: 0.000023966
Epoch [18/100], Loss: 0.000022923
Epoch [19/100], Loss: 0.000021955
Epoch [20/100], Loss: 0.000021063
Epoch [21/100], Loss: 0.000020232
Epoch [22/100], Loss: 0.000019449
Epoch [23/100], Loss: 0.000018713
Epoch [24/100], Loss: 0.000018017
Epoch [25/100], Loss: 0.000017362
Epoch [26/100], Loss: 0.000016741
Epoch [27/100], Loss: 0.000016150
Epoch [28/100], Loss: 0.000015590
Epoch [29/100], Loss: 0.000015056
Epoch [30/100], Loss: 0

In [13]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_forward_6.pth')